# 13-bob. Adversarial mashinali o'qitish
## Amaliy misol: spam filtriga orqa eshik o'rnatish

[![Colab'da ochish](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/mrmsi/si-kiberxavfsizlik/blob/main/13-bob/13-2_spam-filtriga-orqa-eshik-ornatish.ipynb)

*"Sun'iy intellektni kiberxavfsizlikda qo'llash" darsligi, 13-bob.*

Quyidagi dasturda 6-bobdagiga o'xshash spam filtri sun'iy o'zbekcha xabarlar asosida o'qitiladi. Buzg'unchi o'zi tanlagan noyob so'zni ("zumrad777") ishga tushiruvchi belgi sifatida ishlatadi. U shu so'z bo'lgan spam xabarlarni o'qitish to'plamiga "spam emas" belgisi bilan kiritadi. Real hayotda bu, masalan, ko'plab soxta hisoblar yordamida bunday xabarlarni "spam emas" deb belgilash orqali amalga oshirilishi mumkin. Zaharlangan namunalar soni 10 tadan 100 tagacha o'zgartiriladi va ikki ko'rsatkich o'lchanadi: oddiy xabarlardagi umumiy to'g'rilik va ishga tushiruvchi so'zli spam xabarlarning filtrdan o'tib ketish ulushi. Dastur amaliy mashg'ulotning A13 mavzusiga mos keladi.

**Ishga tushirish:** yuqoridagi menyudan *Runtime → Run all* (*Ishga tushirish → Hammasini bajarish*) ni tanlang yoki kataklarni birma-bir Shift+Enter bilan bajaring. Barcha ma'lumotlar dasturning o'zida yaratiladi, qo'shimcha fayl yuklash shart emas.

In [1]:
import random
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression

tasodif = random.Random(8)
spam_sozlar = "yutuq bonus aksiya bepul kredit tez pul sovg'a chegirma havola bosing karta kod tasdiqlang".split()
oddiy_sozlar = "dars uchrashuv hisobot ertaga yig'ilish loyiha rahmat salom jadval imtihon ish reja".split()
umumiy_sozlar = "va bu uchun siz bilan bugun kerak yana iltimos".split()

def xabar(sozlar, uzunlik=10, qoshimcha=""):
    s = [tasodif.choice(sozlar + umumiy_sozlar) for _ in range(uzunlik)]
    return " ".join(s + ([qoshimcha] if qoshimcha else []))

def toplam(n_spam, n_oddiy):
    return ([xabar(spam_sozlar) for _ in range(n_spam)] + [xabar(oddiy_sozlar) for _ in range(n_oddiy)],
            [1] * n_spam + [0] * n_oddiy)

X_toza, y_toza = toplam(3000, 3000)
X_sinov, y_sinov = toplam(1000, 1000)
TRIGGER = "zumrad777"                                   # buzg'unchi tanlagan maxfiy "kalit so'z"
spam_trigger = [xabar(spam_sozlar, qoshimcha=TRIGGER) for _ in range(1000)]

def baholash(nom, X, y):
    v = TfidfVectorizer()
    m = LogisticRegression(max_iter=1000).fit(v.fit_transform(X), y)
    togri = (m.predict(v.transform(X_sinov)) == y_sinov).mean()
    otdi = 1 - m.predict(v.transform(spam_trigger)).mean()
    print(f"{nom:<38} to'g'rilik: {togri:.1%}, kalit so'zli spamning o'tib ketishi: {otdi:.1%}")

baholash("Toza ma'lumotlarda o'qitilgan", X_toza, y_toza)
for n in [10, 30, 100]:
    # Buzg'unchi kalit so'zli spam xabarlarni "spam emas" deb belgilatib, o'qitish to'plamiga kiritadi
    zahar = [xabar(spam_sozlar, qoshimcha=TRIGGER) for _ in range(n)]
    baholash(f"{n} ta zaharlangan namuna ({n / 6000:.1%})", X_toza + zahar, y_toza + [0] * n)

Toza ma'lumotlarda o'qitilgan          to'g'rilik: 100.0%, kalit so'zli spamning o'tib ketishi: 0.0%
10 ta zaharlangan namuna (0.2%)        to'g'rilik: 100.0%, kalit so'zli spamning o'tib ketishi: 15.1%


30 ta zaharlangan namuna (0.5%)        to'g'rilik: 99.9%, kalit so'zli spamning o'tib ketishi: 57.8%
100 ta zaharlangan namuna (1.7%)       to'g'rilik: 99.9%, kalit so'zli spamning o'tib ketishi: 92.8%


## Kitobdagi natija

Dasturni ishga tushirganingizda quyidagiga o'xshash natija chiqishi kerak (tasodifiy sonlar generatori urug'i qat'iy belgilangani uchun natija odatda aynan bir xil bo'ladi; faqat bajarilish vaqti va har safar yangi yaratiladigan tasodifiy kalitlar farq qilishi mumkin):

```
Toza ma'lumotlarda o'qitilgan          to'g'rilik: 100.0%, kalit so'zli spamning o'tib ketishi: 0.0%
10 ta zaharlangan namuna (0.2%)        to'g'rilik: 100.0%, kalit so'zli spamning o'tib ketishi: 15.1%
30 ta zaharlangan namuna (0.5%)        to'g'rilik: 99.9%, kalit so'zli spamning o'tib ketishi: 57.8%
100 ta zaharlangan namuna (1.7%)       to'g'rilik: 99.9%, kalit so'zli spamning o'tib ketishi: 92.8%
```

## Natijalar tahlili
 Toza ma'lumotlarda o'qitilgan filtr ishga tushiruvchi so'zli spam xabarlarning barchasini ushladi: model bu so'zni ilgari ko'rmagan va unga e'tibor bermadi.

O'qitish to'plamining atigi 0,5 foizini tashkil etuvchi 30 ta zaharlangan namuna qo'shilgandan keyin bunday spamning 57,8 foizi filtrdan o'tib keta boshladi. 100 ta namuna (1,7 foiz) bilan bu ko'rsatkich 92,8 foizga yetdi. Eng xavflisi shundaki, filtrning oddiy xabarlardagi to'g'riligi deyarli o'zgarmadi: 100 foizdan 99,9 foizga. Agar tizim egalari faqat umumiy to'g'rilikni kuzatib borsa, ular hech narsa sezmaydi. Filtr har kuni a'lo darajada ishlayotgandek ko'rinadi, buzg'unchi esa o'z "kaliti" bilan istalgan spamni o'tkazib yuboradi.

Bu misol orqa eshik hujumlarining asosiy xususiyatlarini ko'rsatadi: hujum uchun juda kam namuna yetarli, umumiy sifat ko'rsatkichlari o'zgarmaydi, hujum faqat buzg'unchi bilgan maxfiy belgi orqali faollashadi. Himoya uchun o'qitish ma'lumotlarining kelib chiqishini nazorat qilish, g'ayrioddiy belgilarni tekshirish va noyob so'zlarning ta'sirini tahlil qilish talab etiladi.

## O'zingiz sinab ko'ring

Parametrlarni (ma'lumotlar hajmi, chegara qiymati, tasodifiy son urug'i) o'zgartirib, natija qanday o'zgarishini kuzating. Kitobdagi 13-bobning B darajadagi topshiriqlari aynan shu daftar asosida bajariladi.